# DisasterAI — Adaptoru birlestir ve GGUF'a cevir

**Sira onemli:** once birlestirme (Kaggle'in kendi torch/transformers surumuyle),
SONRA llama.cpp kurulumu. Tersi olursa llama.cpp'nin bagimliliklari numpy'i
dusurup ortami bozuyor.

**Disk:** birlesik model (~16 GB) ve ara GGUF (~16 GB) `/kaggle/temp` altina
yazilir; `/kaggle/working` (20 GB sinirli) sadece nihai 4-bit dosyayi tutar.

**Kaggle ayarlari:**
- Accelerator: GPU T4 x2 (ya da None; birlestirme CPU'da yapiliyor)
- Add Input > **Your Work > qlora_egitim**

Cikti: `/kaggle/working/disasterai-Q4_K_M.gguf` (~4.9 GB)

In [ ]:
#@title 1) Ortami hazirla + adaptoru bul
# Kaggle'daki torchao 0.10 eski; PEFT >=0.16 bekliyor ve ImportError firlatiyor.
# LoRA birlestirme icin torchao GEREKMIYOR -> kaldiriyoruz (en hafif cozum).
!pip -q uninstall -y torchao 2>&1 | tail -1

import os, shutil
adaylar = []
for kok, _, dosyalar in os.walk("/kaggle/input"):
    if "adapter_config.json" in dosyalar:
        adaylar.append(kok)
nihai = [a for a in adaylar if "disasterai_lora" in a]
adaptor_yol = nihai[0] if nihai else (sorted(adaylar)[-1] if adaylar else None)
print("adaptor:", adaptor_yol)
if not adaptor_yol:
    raise SystemExit("Adaptor bulunamadi — Add Input > Your Work > qlora_egitim ekle.")

# Gecici alan: /kaggle/temp her imajda yok -> en cok bos yeri olani sec
ADAYLAR = ["/kaggle/temp", "/tmp", "/kaggle/working"]
def bos_gb(p):
    try:
        os.makedirs(p, exist_ok=True)
        s = shutil.disk_usage(p)
        return s.free / 1e9
    except Exception:
        return -1
GECICI = max(ADAYLAR, key=bos_gb)
print(f"gecici alan: {GECICI} ({bos_gb(GECICI):.0f} GB bos)")
for p in ADAYLAR:
    print(f"   {p}: {bos_gb(p):.0f} GB")

In [ ]:
#@title 2) Taban modeli indir + adaptoru BIRLESTIR (GPU'lara dagitarak, ~20 dk)
# CPU RAM'i (~29 GB) 8B fp16 model + merge kopyalari asiyordu.
# Cozum: agirliklari iki T4'un VRAM'ine dagit (2 x 15 GB), CPU'yu bosalt.
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1"
BIRLESIK = f"{GECICI}/birlesik"

print("GPU sayisi:", torch.cuda.device_count())

tokenizer = AutoTokenizer.from_pretrained(adaptor_yol)
taban = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",           # iki GPU'ya bol
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(taban, adaptor_yol)
model = model.merge_and_unload()
print("birlestirme tamam, diske yaziliyor...")

# Kucuk parcalar halinde yaz: tek seferde buyuk tensor kopyasi olusmasin
model.save_pretrained(BIRLESIK, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(BIRLESIK)

del model, taban
gc.collect(); torch.cuda.empty_cache()
print("birlestirildi ->", BIRLESIK)
!du -sh {BIRLESIK}

In [ ]:
#@title 3) llama.cpp kur (ARTIK guvenli — birlestirme bitti)
LLAMA = f"{GECICI}/llama.cpp"
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA} \
 || git clone -q --depth 1 https://github.com/ggerganov/llama.cpp {LLAMA}
!pip -q install -U gguf sentencepiece protobuf 2>&1 | tail -2
print("llama.cpp hazir")
!ls {LLAMA}/convert_hf_to_gguf.py

In [ ]:
#@title 4) GGUF'a cevir (f16, ~10 dk)
F16 = f"{GECICI}/disasterai-f16.gguf"
!python {LLAMA}/convert_hf_to_gguf.py {BIRLESIK} --outfile {F16} --outtype f16
!ls -lh {F16}

In [ ]:
#@title 5) 4-bit sikistir -> /kaggle/working (~10 dk)
!cd {LLAMA} && cmake -B build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF > /dev/null 2>&1 \
 && cmake --build build --target llama-quantize -j4 > /dev/null 2>&1
!ls {LLAMA}/build/bin/llama-quantize
!{LLAMA}/build/bin/llama-quantize {F16} /kaggle/working/disasterai-Q4_K_M.gguf Q4_K_M
!ls -lh /kaggle/working/

In [ ]:
#@title 6) Temizlik ve ozet
import shutil, os
shutil.rmtree(BIRLESIK, ignore_errors=True)
if os.path.exists(F16):
    os.remove(F16)
print("kalan cikti dosyalari:")
!ls -lh /kaggle/working/
print("\n>>> Output sekmesinden disasterai-Q4_K_M.gguf indir (~4.9 GB)")